<a href="https://colab.research.google.com/github/Alimusy/OIBSIP/blob/main/Car_price.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pickle


In [16]:
# --- A. Load the Data ---
df = pd.read_csv('car data.csv')
print("Initial Data Loaded (First 5 Rows):")
display(df.head())
print("-" * 50)

Initial Data Loaded (First 5 Rows):


,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


--------------------------------------------------


In [17]:
# Feature Engineering and Cleaning

# Create the 'Car_Age' feature from 'Year'
current_year = 2025
df['Car_Age'] = current_year - df['Year']

# Drop the original 'Year' column and the 'Car_Name'
df.drop(['Year', 'Car_Name'], axis=1, inplace=True)


# 2. One-Hot Encoding for Categorical Variables
categorical_cols = ['Fuel_Type', 'Selling_type', 'Transmission']
# Use get_dummies and set drop_first=True to prevent multicollinearity
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Features after cleaning and encoding (First 5 Rows):")
display(df.head())
print("-" * 50)

Features after cleaning and encoding (First 5 Rows):


,Selling_Price,Present_Price,Driven_kms,Owner,Car_Age,Fuel_Type_Diesel,Fuel_Type_Petrol,Selling_type_Individual,Transmission_Manual
0,3.35,5.59,27000,0,11,False,True,False,True
1,4.75,9.54,43000,0,12,True,False,False,True
2,7.25,9.85,6900,0,8,False,True,False,True
3,2.85,4.15,5200,0,14,False,True,False,True
4,4.60,6.87,42450,0,11,True,False,False,True


--------------------------------------------------


In [18]:
#Target variable is 'Selling_Price'
X = df.drop('Selling_Price', axis=1)
Y = df['Selling_Price']

# Split the data (80% training, 20% testing)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(f"Training set size (rows): {X_train.shape[0]}")
print(f"Testing set size (rows): {X_test.shape[0]}")
print("-" * 50)

Training set size (rows): 240
Testing set size (rows): 61
--------------------------------------------------


In [19]:
#Model Training

# Initialize and train the Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, Y_train)


RandomForestRegressor(random_state=42)

In [21]:
#Prediction and Evaluation ---

# Make predictions on the test set
predictions = rf_model.predict(X_test)

# Calculate key evaluation metrics
mae = mean_absolute_error(Y_test, predictions)
r2 = r2_score(Y_test, predictions)

# Print the evaluation metrics
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Mean Absolute Error (MAE): 0.64
R-squared (R2): 0.96


In [23]:
# Get the list of features the model expects (must be in the exact order)
model_features = X_train.columns
print("Model expects features in this order:")
print(model_features.tolist())

# Save the trained Random Forest model
pickle.dump(rf_model, open('final_car_model.pkl', 'wb'))

Model expects features in this order:
['Present_Price', 'Driven_kms', 'Owner', 'Car_Age', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Selling_type_Individual', 'Transmission_Manual']


In [27]:
# Get feature importances from the trained model
feature_importances = pd.Series(rf_model.feature_importances_, index=model_features).sort_values(ascending=False)

print("\n--- Feature Importances ---")
print('% Influence of all the training features on the outcome of the model')
display(feature_importances)


--- Feature Importances ---
% Influence of all the training features on the outcome of the model


,0
Present_Price,0.881166
Car_Age,0.059495
Driven_kms,0.040265
Transmission_Manual,0.009646
Fuel_Type_Diesel,0.004339
Fuel_Type_Petrol,0.002564
Selling_type_Individual,0.002155
Owner,0.000370


In [28]:
def predict_car_price(model, present_price, driven_kms, owner, year, fuel_type, selling_type, transmission):

    data = {
        'Present_Price': [present_price],
        'Driven_kms': [driven_kms],
        'Owner': [owner],
        'Year': [year],
        'Fuel_Type': [fuel_type],
        'Selling_type': [selling_type],
        'Transmission': [transmission]
    }

    input_df = pd.DataFrame(data)

    # FEATURE ENGINEERING (Calculate Car_Age)
    input_df['Car_Age'] = current_year - input_df['Year']
    input_df.drop('Year', axis=1, inplace=True)

    # ONE-HOT ENCODING (Replicate training data steps)

    # Create the full list of columns expected by the model (excluding the original encoded ones)
    full_features = model_features.tolist()

    # Initialize encoded columns
    input_df['Fuel_Type_Diesel'] = 0
    input_df['Fuel_Type_Petrol'] = 0
    input_df['Selling_type_Individual'] = 0
    input_df['Transmission_Manual'] = 0

    if fuel_type.lower() == 'petrol':
        input_df['Fuel_Type_Petrol'] = 1
    elif fuel_type.lower() == 'diesel':
        input_df['Fuel_Type_Diesel'] = 1

    if selling_type.lower() == 'individual':
        input_df['Selling_type_Individual'] = 1

    if transmission.lower() == 'manual':
        input_df['Transmission_Manual'] = 1

    # Standardize input to match the feature set (e.g., fill in the 0s for missing dummies)
    for col in full_features:
        if col not in input_df.columns:
            input_df[col] = 0

    # Final Data Preparation and Prediction

    # Select and order the columns exactly as the model expects
    final_input = input_df[full_features]

    # Make the prediction
    predicted_price = model.predict(final_input)[0]

    return predicted_price

# Get User Input for a new car and make a prediction

print("=============================================")
print("   ENTER NEW CAR DETAILS FOR PREDICTION")
print("=============================================")

# Get input from the user
try:
    user_present_price = float(input("Enter Present Price (Showroom price in Lakhs): "))
    user_driven_kms = int(input("Enter Kilometers Driven: "))
    user_owner = int(input("Enter Number of Owners (0 = First Owner, 1 = Second, etc.): "))
    user_year = int(input("Enter Year of Manufacture (e.g., 2020): "))
    user_fuel = input("Enter Fuel Type (Petrol, Diesel, or CNG): ")
    user_seller = input("Enter Selling Type (Dealer or Individual): ")
    user_trans = input("Enter Transmission Type (Manual or Automatic): ")

    # Make the prediction using user input
    predicted_value = predict_car_price(
        rf_model,
        user_present_price,
        user_driven_kms,
        user_owner,
        user_year, # Keep year input for Car_Age calculation
        user_fuel,
        user_seller,
        user_trans
    )

    print("\n=============================================")
    print("         NEW CAR PRICE PREDICTION")
    print("=============================================")
    print(f"Input Car Present Price: ${user_present_price} Lakhs")
    print(f"Input Car Age: {current_year - user_year} years") # Calculate age based on user input year
    print(f"Predicted Selling Price: **${predicted_value:.2f} Lakhs**")
    print("=============================================")

except ValueError:
    print("\n=============================================")
    print("             INPUT ERROR")
    print("=============================================")
    print("Please ensure you enter valid numbers for numerical inputs.")
    print("=============================================")
except Exception as e:
    print(f"An error occurred: {e}")

   ENTER NEW CAR DETAILS FOR PREDICTION
Enter Present Price (Showroom price in Lakhs): 7
Enter Kilometers Driven: 23453
Enter Number of Owners (0 = First Owner, 1 = Second, etc.): 0
Enter Year of Manufacture (e.g., 2020): 2020
Enter Fuel Type (Petrol, Diesel, or CNG): petrol
Enter Selling Type (Dealer or Individual): dealer
Enter Transmission Type (Manual or Automatic): manual

         NEW CAR PRICE PREDICTION
Input Car Present Price: $7.0 Lakhs
Input Car Age: 5 years
Predicted Selling Price: **$5.45 Lakhs**
